# NewEra stellar spectra: build and validation

The stellar SED of the pipeline (`astro_utils.STELLAR_MODEL = "newera"`) is the PHOENIX/1D **NewEra** grid (Hauschildt et al. 2025, A&A 698, A47), V3 JWST product: 0.6–28.5 µm at 0.2 nm, data DOI 10.25592/uhhfdm.18108. It enters only the thermal $F_p/F_\star$ normalisation (SPIDERMAN's stellar grid) and `T_bright`; the bolometric insolation ($\sigma T_{\rm eff}^4$) and the grey reflected curve do not depend on it.

This notebook converts the 12 GB archive into the compact cube that `newera_utils.newera_spectrum` reads, then validates it. Section 1 runs once; the remaining sections are checks that can be re-run at any time.

**Data** (gitignored, `Stellar Model/NewEra/`):
- `raw/PHOENIX-NewEraV3-JWST-SPECTRA.tar.gz` (md5 `8a445ad80d3573c751b27c9a3d959cd8`), read as a stream, never extracted;
- `raw/PHOENIX-NewEraV3-add001-JWST-SPECTRA.Z+0.5.txt`, models added after the paper (they only fill gaps of the main archive);
- `raw/hsr/*.h5`, six full-range single models (900 Å – 930 µm), used for validation only.

**Cube:** α = 0 (scaled solar), $T_{\rm eff}$ = 2300–7200 K (100 K), log g = 3.5–6.0 (0.5), [M/H] = −1.5 … +0.5 (0.5), rebinned flux-conservingly to R = 5000. Interpolation between models is linear in flux (as stsynphot's `grid_to_spec`); a missing neighbour raises instead of being skipped. Widen `CUBE_*` in `newera_utils` and rebuild if a star falls outside.

In [ ]:
import glob, re, json
import numpy as np
import h5py
import matplotlib.pyplot as plt
import contextlib, io
with contextlib.redirect_stdout(io.StringIO()):   # mute "no fspidermanrc file detected"
    import fspiderman as sp
    import hyperillumination_phasecurve_utils as hpu
import astro_utils as au
import newera_utils as nu
from HItargets_utils import filter_lambda, filter_loc, hi_planets_by_name, PAPER_PLANETS
from paper_plot_utils import paper_figure

SB = au.STEFAN_BOLTZMANN
BANDS = ["JWST_NIRSpec.G395H_NRS1", "JWST_NIRSpec.G395H_NRS2", "Spitzer/IRAC2", "MIRI/LRS", "MIRI/F1280W", "MIRI/F1500W", "MIRI/F1800W"]
SHORT = {b: b.split("_")[-1].split("/")[-1] for b in BANDS}
FILT = {b: sp.get_filter(filter_loc(b)) for b in BANDS}

def band(lam, F, b):
    '''Photon-weighted in-band integral of F [any per-wavelength unit] with the real throughput, as stellar_spidergrid integrates the star.'''
    l1, l2 = filter_lambda(b); fw, fr = FILT[b]
    m = (lam > l1) & (lam < l2); x = np.concatenate(([l1], lam[m], [l2])); y = np.interp(x, lam, F)
    return np.trapezoid(y * np.interp(x, fw, fr), x)

def k_factor(lam, F, T, b):
    '''Depth multiplier k = S_BB / S_model: a thermal Fp/Fs with this stellar SED is k times the blackbody-star value. F is the surface flux [W m^-3].'''
    return band(lam, np.pi * au.planck_spectrum(lam, T), b) / band(lam, F, b)

## 1. Build the cube (once, about 15 min)

In [ ]:
if not nu.CUBE_FLUX.exists():
    nu.build_cube()
wave, tg, gg, zg, cube = nu._cube()
prov = json.loads(str(np.load(nu.CUBE_AXES)["provenance"]))
print(f"cube {cube.shape} = ([M/H], log g, T_eff, wavelength), built {prov['built']} from {prov['source']} + {prov['additions']}")
print(f"wavelengths {wave[0]*1e6:.3f}-{wave[-1]*1e6:.2f} um ({wave.size} bins); missing models: {prov['missing'] if prov['missing'] else 'none'}")

## 2. Normalisation: full-range models vs $\sigma T_{\rm eff}^4$

The JWST product starts at 0.6 µm, so the bolometric check uses the full-range HSR files of six representative models (beyond their last point a Rayleigh–Jeans tail is added). Each file also stores the Planck function at $T_{\rm eff}$; its integral must be 1 in units of $\sigma T_{\rm eff}^4$, which checks the flux units. What to look for: model ratios within about a per cent of unity. Spherical, lower-gravity models can sit slightly lower.

In [ ]:
HSR = {}
for f in sorted(glob.glob(str(nu.NEWERA_HSR / "*.h5"))):
    m = re.search(r"lte(\d{5})-(\d\.\d\d)([-+]\d\.\d)\.", f); key = (float(m.group(1)), float(m.group(2)), float(m.group(3)))
    with h5py.File(f, "r") as h:   # stored as log10, erg s^-1 cm^-2 cm^-1 -> W m^-3 (x 0.1)
        wl = h["/PHOENIX_SPECTRUM/wl"][()] * 1e-10; F = 10. ** h["/PHOENIX_SPECTRUM/flux"][()] * 0.1; bb = 10. ** h["/PHOENIX_SPECTRUM/bb"][()] * 0.1
    o = np.argsort(wl); HSR[key] = (wl[o], F[o])
    T = key[0]; tail = F[o][-1] * wl[o][-1] / 3.
    print(f"T_eff = {T:.0f} K, log g = {key[1]}, [M/H] = {key[2]:+.1f}:  (int F + tail) / sigma T^4 = {(np.trapezoid(F[o], wl[o]) + tail)/(SB*T**4):.4f}   "
          f"int B / sigma T^4 = {np.trapezoid(bb[o], wl[o])/(SB*T**4):.4f}")

## 3. The JWST product reproduces the full-range spectra

Band-integrated flux of the cube (rebinned JWST product) over the full-range file of the same model, for every instrument band. The cube is a rebinned copy of the same models, so the ratios should be 1 to about $10^{-3}$.

In [ ]:
for (T, g, z), (wl, F) in HSR.items():
    Fc = np.asarray(cube[nu._index(z, zg), nu._index(g, gg), nu._index(T, tg)], float) * 1e9   # W m^-2 nm^-1 -> W m^-3
    print(f"T_eff = {T:.0f}, log g = {g}, [M/H] = {z:+.1f}:  " + "  ".join(f"{SHORT[b]} {band(wave, Fc, b)/band(wl, F, b):.4f}" for b in BANDS))

## 4. Screening of every model in the cube

For every model and band, $k = S_{\rm BB}/S_{\rm model}$, the factor by which the stellar SED rescales thermal depths relative to a blackbody star at the same $T_{\rm eff}$. A broken model (wrong normalisation, a mislabelled spectrum, a placeholder) shows up as an isolated kink in $k$ along $T_{\rm eff}$, so kinks larger than 3 % in $\Delta^2 \ln k$ are flagged, as are 10 µm fluxes far from the blackbody value.

How to read the flags: a corrupt file gives an isolated flag at one model; flags that sit at the same $T_{\rm eff}$ for many log g and metallicities instead mark a systematic seam in how the grid's coolest models were computed (at the time of writing they cluster at the cool end, below about 2900 K, mostly at low metallicity and high log g). Either way, what matters is whether a host leans on a flagged model: the last cell of this section prints, for every host, the interpolation weight that flagged models carry at its $T_{\rm eff}$. The figure shows $k$ against $T_{\rm eff}$ for all metallicities.

In [ ]:
kcube = np.full(cube.shape[:3] + (len(BANDS),), np.nan)
flags = []
for k in range(zg.size):
    for j in range(gg.size):
        for i in range(tg.size):
            F = np.asarray(cube[k, j, i], float) * 1e9
            if not np.isfinite(F[0]): continue
            kcube[k, j, i] = [k_factor(wave, F, tg[i], b) for b in BANDS]
            r10 = np.interp(10e-6, wave, F) / (np.pi * au.planck_spectrum(10e-6, tg[i]))
            if not 0.6 < r10 < 1.2: flags.append(f"F(10 um)/pi B = {r10:.2f} at T_eff={tg[i]:.0f}, log g={gg[j]}, [M/H]={zg[k]:+.1f}")
        for n, b in enumerate(BANDS):
            y = np.log(kcube[k, j, :, n]); d2 = y[:-2] - 2 * y[1:-1] + y[2:]
            flags += [f"kink in k({SHORT[b]}) at T_eff={tg[i+1]:.0f}, log g={gg[j]}, [M/H]={zg[k]:+.1f}: {d2[i]:+.3f}" for i in np.flatnonzero(np.abs(d2) > 0.03)]
print(f"{np.isfinite(kcube[..., 0]).sum()} models screened, {len(flags)} flags")
for f in flags: print("  ", f)

In [ ]:
HOSTS = [*PAPER_PLANETS, "LHS 3844 b", "TRAPPIST-1 b", "GJ 367 b"]   # the planets of the paper and three cool hosts
flagged = {(float(t), float(g), float(z)) for t, g, z in re.findall(r"T_eff=(\d+), log g=([\d.]+), \[M/H\]=([+-][\d.]+)", "\n".join(flags))}
for name in HOSTS:
    pl = hi_planets_by_name(name)
    (i0, i1, wi), (j0, j1, wj), (k0, k1, wk) = nu._bracket(pl.T2_eff, tg, "T_eff"), nu._bracket(pl.logg, gg, "logg"), nu._bracket(pl.feh, zg, "[M/H]")
    w = sum(wz * wg * wt for k, wz in ((k0, 1 - wk), (k1, wk)) for j, wg in ((j0, 1 - wj), (j1, wj)) for i, wt in ((i0, 1 - wi), (i1, wi)) if (tg[i], gg[j], zg[k]) in flagged)
    print(f"{name:17s} weight of flagged models at its T_eff: {w:.3f}")

In [ ]:
PAPER_MODE = False
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharex=True)
for ax, b in zip(axes, ["JWST_NIRSpec.G395H_NRS1", "MIRI/LRS", "MIRI/F1500W"]):
    n = BANDS.index(b)
    for k, z in enumerate(zg):
        for j, g, ls in ((nu._index(4.5, gg), 4.5, "-"), (nu._index(5.0, gg), 5.0, "--")):
            ax.plot(tg, kcube[k, j, :, n], ls, color=plt.cm.viridis(k / (zg.size - 1)), label=f"[M/H] = {z:+.1f}" if g == 4.5 else None)
    ax.axhline(1, color="0.5", lw=0.8); ax.set(title=SHORT[b], xlabel=r"$T_{\rm eff}$ [K]"); ax.grid(alpha=0.3)
axes[0].set_ylabel(r"$k = S_{\rm BB}/S_{\rm NewEra}$  (solid log g 4.5, dashed 5.0)"); axes[0].legend(fontsize=8)
fig.suptitle("NewEra: depth multiplier relative to a blackbody star")
fig.tight_layout()
paper_figure("stellar_newera_k_teff", PAPER_MODE)
plt.show()

## 5. Comparison with solar BT-Settl

$k$ from NewEra and from the solar-metallicity BT-Settl models of the stsynphot catalog (renormalised to $\sigma T_{\rm eff}^4$), at log g = 4.5. Two independent model atmospheres: agreement to a few per cent in the mid-IR is expected; larger differences for M dwarfs in the near-IR reflect molecular opacities, not a bug.

In [ ]:
import stsynphot
for T in [3000., 3500., 4100., 4600., 5500., 6300.]:
    nw, nI = au.stellar_spectrum(T, 4.5, 0.0, "newera")
    s = stsynphot.grid_to_spec("phoenix", T, 0.0, 4.5); w = s.waveset; bw = w.to_value("m"); bF = s(w, flux_unit="flam").value * 1e7
    bF *= SB * T**4 / np.trapezoid(bF, bw)
    print(f"T_eff = {T:.0f} K  k NewEra / BT-Settl:  " + "  ".join(f"{SHORT[b]} {k_factor(nw, np.pi*nI, T, b):.3f}/{k_factor(bw, bF, T, b):.3f}" for b in BANDS))

## 6. Metallicity

How much [M/H] changes $k$ relative to solar at log g = 4.5, the effect that motivated NewEra. The figure shows the surface flux relative to the blackbody for two temperatures and the cube's metallicities.

In [ ]:
for T in [3000., 3500., 4100., 4600., 5500.]:
    i, j, k0 = nu._index(T, tg), nu._index(4.5, gg), nu._index(0.0, zg)
    print(f"T_eff = {T:.0f} K:  " + " | ".join(f"{SHORT[b]} " + " ".join(f"{z:+.1f}: {kcube[nu._index(z, zg), j, i, BANDS.index(b)]/kcube[k0, j, i, BANDS.index(b)]-1:+.1%}" for z in zg if z != 0.0)
                                        for b in ["JWST_NIRSpec.G395H_NRS1", "MIRI/LRS", "MIRI/F1500W"]))

In [ ]:
PAPER_MODE = False
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
edges_nm, centres_nm = nu.cube_bins(R=300, lo=1000., hi=20000.)   # binned to R = 300 for display only
for ax, T in zip(axes, [3500., 4600.]):
    for k, z in enumerate(zg):
        w, I = au.stellar_spectrum(T, 4.5, z, "newera")
        ratio = nu.rebin(w, I, edges_nm * 1e-9) / nu.rebin(w, au.planck_spectrum(w, T), edges_nm * 1e-9)
        ax.plot(centres_nm * 1e-3, ratio, lw=1.2, color=plt.cm.viridis(k / (zg.size - 1)), label=f"[M/H] = {z:+.1f}")
    ax.axhline(1, color="0.5", lw=0.8); ax.set(xscale="log", title=rf"$T_{{\rm eff}}$ = {T:.0f} K, log g = 4.5", xlabel=r"wavelength [$\mu$m]"); ax.grid(alpha=0.3)
axes[0].set_ylabel(r"$F_\lambda / \pi B_\lambda(T_{\rm eff})$"); axes[0].legend(fontsize=8)
fig.suptitle("NewEra surface flux relative to a blackbody (binned to R = 300)")
fig.tight_layout()
paper_figure("stellar_newera_metallicity", PAPER_MODE)
plt.show()

## 7. All hosts through the pipeline

`stellar_spidergrid` with every host's own $(T_{\rm eff}, \log g, [{\rm Fe/H}])$, and $k$ at the host's $T_{\rm eff}$ from SPIDERMAN's node spline: the factor by which the thermal depths of that system differ from those with a blackbody star. What to look for: every host builds; $k$ above 1 in the mid-IR (stellar mid-IR flux below the blackbody), largest for the coolest hosts.

In [ ]:
for name in HOSTS:
    pl = hi_planets_by_name(name)
    row = []
    for b in ["JWST_NIRSpec.G395H_NRS1", "MIRI/LRS", "MIRI/F1500W"]:
        g_ne = hpu.stellar_spidergrid(filter_lambda(b), filter_loc(b), pl.logg, pl.feh, stellar_model="newera")
        g_bb = hpu.stellar_spidergrid(filter_lambda(b), filter_loc(b), pl.logg, pl.feh, stellar_model="blackbody")
        row.append(f"{SHORT[b]} {hpu.spidergrid_value(g_bb, pl.T2_eff)/hpu.spidergrid_value(g_ne, pl.T2_eff):.3f}")   # SPIDERMAN's own spline through the nodes
    print(f"{name:17s} T_eff {pl.T2_eff:5.0f} K, log g {pl.logg:.2f}, [Fe/H] {pl.feh:+.2f}:  k  " + "  ".join(row))